In [0]:
import kagglehub, os

# Download the dataset to the driver's local disk
path = kagglehub.dataset_download("computingvictor/transactions-fraud-datasets")
print("Path to dataset files:", path)

# Show files and sizes (look for transactions_data.csv, ~1.2 GB)
for f in os.listdir(path):
    print(f, round(os.path.getsize(os.path.join(path, f)) / 1e6, 1), "MB")

In [0]:
# Create a Volume for raw files and copy the CSV from the driver's local disk
spark.sql("CREATE VOLUME IF NOT EXISTS payments.default.raw_files")

# dbutils.fs cannot access non-/Workspace local paths on serverless compute;
# plain Python file IO (shutil) can, and can write directly into UC Volumes
import shutil
shutil.copyfile(
    f"{path}/transactions_data.csv",
    "/Volumes/payments/default/raw_files/transactions_data.csv"
)

In [0]:
%sql
-- Create the full raw table from the CSV in the UC Volume
CREATE TABLE IF NOT EXISTS payments.default.pay_transactions_full
AS SELECT * FROM read_files(
  '/Volumes/payments/default/raw_files/transactions_data.csv',
  format => 'csv',
  header => true
);

-- Expect ~13M rows and dates from 2010 to 2019
SELECT COUNT(*) AS n_rows, MIN(CAST(date AS TIMESTAMP)) AS first_tx, MAX(CAST(date AS TIMESTAMP)) AS last_tx
FROM payments.default.pay_transactions_full;

In [0]:
%sql
-- Replace the old 2010-only table with the full data.
-- The old version stays in Delta history, so it can be restored with RESTORE TABLE if needed.
CREATE OR REPLACE TABLE payments.default.pay_transactions AS
SELECT * FROM payments.default.pay_transactions_full;

In [0]:
%sql
-- Row count, period and label coverage after the replacement
SELECT
    COUNT(*)  AS n_tx,
    MIN(date) AS first_tx,
    MAX(date) AS last_tx,
    (SELECT COUNT(*) FROM payments.default.pay_labels l
     LEFT ANTI JOIN payments.default.pay_transactions t ON t.id = l.transaction_id) AS labels_without_tx
FROM payments.default.pay_transactions;

In [0]:
%sql
-- Remove the temporary copy (optional)
DROP TABLE IF EXISTS payments.default.pay_transactions_full;